# Neural Network SMS Text Classifier

A neural classifier uses unigram and bigram TF-IDF features. The vocabulary is learned only from training messages, and a separate training-validation split controls early stopping. The provided holdout file is reserved for final evaluation.

Prepared with AI assistance for an agentic AI class assignment. Starter notebook and original challenge tests: [freeCodeCamp](https://github.com/freeCodeCamp/boilerplate-neural-network-sms-text-classifier). The supplied final test cell is preserved. Run all cells from top to bottom.

In [1]:
import os
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '2')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '2')
DATA_DIR = Path(os.environ.get('FCC_DATA_DIR', 'data')).resolve()
ARTIFACT_DIR = Path(os.environ.get('FCC_ARTIFACT_DIR', 'artifacts')).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def data_file(name, relative_url, extract=False):
    # The official freeCodeCamp CDN repository is a public mirror of the datasets.
    path = DATA_DIR / name
    if not path.exists() or not path.stat().st_size:
        url = 'https://raw.githubusercontent.com/freeCodeCamp/cdn/main/build/project-data/' + relative_url
        with urlopen(url, timeout=120) as response:
            contents = response.read()
        temporary = path.with_suffix(path.suffix + '.part')
        temporary.write_bytes(contents)
        temporary.replace(path)
    if extract:
        with ZipFile(path) as archive:
            for member in archive.namelist():
                (DATA_DIR / member).resolve().relative_to(DATA_DIR)
            archive.extractall(DATA_DIR)
    return path

In [2]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
tf.keras.utils.set_random_seed(42)
try:
    tf.config.threading.set_intra_op_parallelism_threads(2)
    tf.config.threading.set_inter_op_parallelism_threads(2)
except RuntimeError:
    pass  # The runtime may already be initialized when a notebook cell is rerun.

def batches(x, y=None, training=False, batch_size=64):
    contents = x if y is None else (x, y)
    ds = tf.data.Dataset.from_tensor_slices(contents)
    if training:
        ds = ds.shuffle(len(x), seed=42)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 2
    options.threading.max_intra_op_parallelism = 2
    return ds.batch(batch_size).with_options(options).prefetch(1)

In [3]:
train_file_path = data_file('train-data.tsv', 'sms/train-data.tsv')
test_file_path = data_file('valid-data.tsv', 'sms/valid-data.tsv')

In [4]:
training = pd.read_csv(train_file_path, sep='	', header=None,
                       names=['label','message'], keep_default_na=False)
holdout = pd.read_csv(test_file_path, sep='	', header=None,
                      names=['label','message'], keep_default_na=False)
y = training['label'].map({'ham':0,'spam':1}).to_numpy(dtype='float32')
train_rows, validation_rows = train_test_split(np.arange(len(training)),
    test_size=0.15, random_state=42, stratify=y)
train_text = tf.constant(training.iloc[train_rows]['message'].tolist())
validation_text = tf.constant(training.iloc[validation_rows]['message'].tolist())
holdout_text = tf.constant(holdout['message'].tolist())
holdout_y = holdout['label'].map({'ham':0,'spam':1}).to_numpy(dtype='float32')
vectorizer = keras.layers.TextVectorization(max_tokens=12000, ngrams=2,
    standardize='lower_and_strip_punctuation', output_mode='tf_idf')
vectorizer.adapt(batches(train_text))
model = keras.Sequential([
    keras.Input(shape=(), dtype=tf.string), vectorizer,
    keras.layers.Dense(16, activation='relu', kernel_regularizer=keras.regularizers.l2(0.0001)),
    keras.layers.Dropout(0.25), keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer=keras.optimizers.Adam(0.001), loss='binary_crossentropy',
    metrics=['accuracy',keras.metrics.AUC(name='auc')])
counts = np.bincount(y[train_rows].astype(int))
class_weights = {index:len(train_rows)/(2*int(count)) for index,count in enumerate(counts)}

In [5]:
history = model.fit(batches(train_text, y[train_rows], training=True),
    validation_data=batches(validation_text, y[validation_rows]),
    epochs=30, verbose=2, class_weight=class_weights,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', mode='min',
        patience=5, restore_best_weights=True)])
holdout_metrics = model.evaluate(batches(holdout_text, holdout_y), verbose=0, return_dict=True)
sms_accuracy = holdout_metrics['accuracy']
sms_auc = holdout_metrics['auc']
print('Unseen holdout metrics:', holdout_metrics)
holdout_probabilities = np.concatenate([model(x, training=False).numpy().ravel()
                                       for x in batches(holdout_text)])
print(classification_report(holdout_y, holdout_probabilities >= 0.5,
                            target_names=['ham','spam'], zero_division=0))

/workspace/scratch/19945c1fc1e3/ml_runtime/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
Epoch 1/30
56/56 - 1s - 19ms/step - accuracy: 0.8514 - auc: 0.8675 - loss: 0.4696 - val_accuracy: 0.8900 - val_auc: 0.9921 - val_loss: 0.3070
Epoch 2/30
56/56 - 0s - 4ms/step - accuracy: 0.9710 - auc: 0.9935 - loss: 0.1298 - val_accuracy: 0.9410 - val_auc: 0.9924 - val_loss: 0.2000
Epoch 3/30
56/56 - 0s - 4ms/step - accuracy: 0.9927 - auc: 0.9993 - loss: 0.0660 - val_accuracy: 0.9665 - val_auc: 0.9892 - val_loss: 0.1258
Epoch 4/30
56/56 - 0s - 6ms/step - accuracy: 0.9966 - auc: 0.9998 - loss: 0.0432 - val_accuracy: 0.9665 - val_auc: 0.9904 - val_loss: 0.1135
Epoch 5/30
56/56 - 0s - 4ms/step - accuracy: 0.9983 - auc: 1.0000 - loss: 0.

In [6]:
def predict_message(pred_text):
    if not isinstance(pred_text, str):
        raise TypeError('pred_text must be a string')
    probability = float(model(tf.constant([pred_text]), training=False).numpy()[0,0])
    return [probability, 'spam' if probability >= 0.5 else 'ham']

print(predict_message('how are you doing today?'))

[0.005953933112323284, 'ham']


In [7]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()


You passed the challenge. Great job!


In [8]:
assert sms_accuracy >= 0.95, f'Unexpectedly low held-out accuracy: {sms_accuracy:.3f}'
model.save(ARTIFACT_DIR / 'sms_classifier.keras')
plt.plot(history.history['accuracy'], label='training')
plt.plot(history.history['val_accuracy'], label='validation')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()
print(f'PASS: held-out accuracy = {sms_accuracy:.2%}; AUC = {sms_auc:.4f}')

PASS: held-out accuracy = 98.35%; AUC = 0.9931
